# Option Pricing

In [7]:
import csv
from io import StringIO
import requests
import yfinance as yf
import math

## Fetch Real Data

In [10]:
# 1) Fetch 1 year of AAPL daily data from yfinance
aapl = yf.Ticker("AAPL")
aapl_hist = aapl.history(period="1y", interval="1d")

# Spot price
S0 = float(aapl_hist["Close"].dropna().iloc[-1])

# Annualized volatility
log_returns = (aapl_hist["Close"] / aapl_hist["Close"].shift(1)).apply(math.log).dropna()
annualized_volatility = float(log_returns.std() * math.sqrt(252))

# 2) Fetch 3-month T-bill rate (TB3MS) from FRED
# No API key needed: use FRED public CSV endpoint

csv_url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=TB3MS"
resp = requests.get(csv_url, timeout=20)
resp.raise_for_status()
rows = list(csv.DictReader(StringIO(resp.text)))

tb3ms_value = None
for row in reversed(rows):
    value = row.get("TB3MS")
    if value not in (None, "", "."):
        tb3ms_value = float(value)
        break
if tb3ms_value is None:
    raise ValueError("No valid TB3MS value found in FRED CSV response.")
rf = tb3ms_value / 100.0

# 3) Fetch AAPL option chain from yfinance
# This creates a dictionary keyed by expiration date
option_chain = {}
for exp in aapl.options:
    chain = aapl.option_chain(exp)
    option_chain[exp] = {
        "calls": chain.calls.copy(),
        "puts": chain.puts.copy()
    }

print(f"AAPL history rows: {len(aapl_hist)}")
print(f"AAPL option expiries: {len(option_chain)}")
print(f"S0 (last AAPL close): {S0:.4f}")
print(f"rf (TB3MS as decimal): {rf:.6f}")

AAPL history rows: 251
AAPL option expiries: 24
S0 (last AAPL close): 273.0500
rf (TB3MS as decimal): 0.036100


## Set Parameters

In [11]:
# Parameters
T = 0.5  # 6 months in years
K = float(round(S0 * 1.02, 2))  # strike set 2% above spot
sigma = annualized_volatility

## Black-Scholes

In [12]:
# Choose what to price: "call" or "put"
option_to_price = "call"


def norm_cdf(x: float) -> float:
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def bs_option_price(S: float, K: float, T: float, r: float, sigma: float, option_type: str) -> float:
    d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)

    if option_type == "call":
        return S * norm_cdf(d1) - K * math.exp(-r * T) * norm_cdf(d2)
    if option_type == "put":
        return K * math.exp(-r * T) * norm_cdf(-d2) - S * norm_cdf(-d1)
    raise ValueError('option_type must be "call" or "put".')


option_price = bs_option_price(S0, K, T, rf, sigma, option_to_price)

print(f"S0: {S0:.4f}")
print(f"rf: {rf:.6f}")
print(f"sigma (annualized): {sigma:.4f}")
print(f"T (years): {T:.4f}")
print(f"K: {K:.2f}")
print(f"Black-Scholes {option_to_price.capitalize()} Price: {option_price:.4f}")

S0: 273.0500
rf: 0.036100
sigma (annualized): 0.2343
T (years): 0.5000
K: 278.51
Black-Scholes Call Price: 17.8060
